## Load an Ontology

The sample ontology can be found [here](http://www.lesfleursdunormal.fr/static/_downloads/pizza_onto.owl)

In [1]:
from owlready2 import *

In [2]:
# Load the pizza ontology from URL
# This ontology contains classess like Pizza, Topping, etc.
onto = get_ontology("http://www.lesfleursdunormal.fr/static/_downloads/pizza_onto.owl").load()

## Accessing the content of an ontology

In [3]:
# List all classes defined in the ontology
# Classess represent concepts (e.g. Pizza, Topping, etc.)
classes = list(onto.classes())
print("Classes:", classes)

# List all individuals (instances) defined in the ontology
# Individuals are concrete instances of classes
individuals = list(onto.individuals())
print("Individuals:", individuals)

# List all object properties (relationships between individuals)
# has_topping relates Pizza to Topping
object_properties = list(onto.object_properties())
print("Object Properties:", object_properties)

Classes: [pizza_onto.CheeseTopping, pizza_onto.FishTopping, pizza_onto.MeatTopping, pizza_onto.Pizza, pizza_onto.TomatoTopping, pizza_onto.Topping]
Individuals: []
Object Properties: [pizza_onto.has_topping]


## Simple Queries

In [4]:
# Search for entities with IRI ending in 'Topping'
# IRI = Internationlized Resource Identifier (like URI/URL)
toppings = onto.search(iri =  '*Topping')
print("Entities ending with 'Topping':", toppings)

# Find all individuals that have ANY topping relationship
# '*' wildcard matches any object
pizzas_with_toppings = onto.search(has_topping = "*")
print("Pizzas with Toppings:", pizzas_with_toppings)

Entities ending with 'Topping': [pizza_onto.CheeseTopping, pizza_onto.FishTopping, pizza_onto.MeatTopping, pizza_onto.TomatoTopping, pizza_onto.Topping]
Pizzas with Toppings: []


In [5]:
# Create a new pizza instance with toppings
# This adds a new individual to the ontology
my_pizza = onto.Pizza("my_perfect_pizza",
                      has_topping=[onto.CheeseTopping(), onto.MeatTopping()])

# Verify the new pizza appears in searches
updated_pizzas = onto.search(has_topping = "*")
print("Updated Pizzas:", updated_pizzas)

# Access the toppings of our new pizza
print("My pizza toppings:", my_pizza.has_topping)

# Nested search: Find all pizza instances that have cheese topping
# Combines class filtering with property filtering
cheese_pizzas = onto.search(is_a = onto.Pizza,
                            has_topping = onto.search(is_a = onto.CheeseTopping))
print("Cheese Pizzas:", cheese_pizzas)

# Iterate through all instances of Pizza class
print("\nAll Pizza instances:")

for i in onto.Pizza.instances():
  print(f" - {i}")

# Get all properties defined for a specific individual
my_pizza_props = my_pizza.get_properties()
print("Properties of my pizza:", my_pizza_props)

Updated Pizzas: [pizza_onto.my_perfect_pizza]
My pizza toppings: [pizza_onto.cheesetopping1, pizza_onto.meattopping1]
Cheese Pizzas: [pizza_onto.my_perfect_pizza]

All Pizza instances:
 - pizza_onto.my_perfect_pizza
Properties of my pizza: {pizza_onto.has_topping}


## Ontology Classes and Properties

In [6]:
# Use 'with onto:' context to modify the ontology namespace
with onto:
  # Create new ontology class by subclassing Thing
  # Thing is the base class for all ontology objects
  class Restaurant(Thing):
    """Represents a restaurant in the ontology"""
    pass

  # For comparison: Regular Python class (NOT an ontology class)
  class Restaurant2():
    """This is NOT part of the ontology"""
    pass

  # Create object property to relate Restaurant to Pizza
  class servesPizza(ObjectProperty):
    """Relationship: Restaurant serves Pizza"""
    domain = [Restaurant] # Subject must be Restaurant
    range = [onto.Pizza] # Object must be Pizza

  # Create subclass of Pizza with restrictions
  class PizzaWithToppings(onto.Pizza):
    """Pizza that must have at least 1 topping"""
    # equivalent_to defines logical equivalence using restrictions
    equivalent_to = [
      # Pizza AND has at least 1 Topping
      onto.Pizza & onto.has_topping.min(1, onto.Topping)
      # Other restrictions: .some(), .only(), .max(), .exactly()
    ]

    def who(self):
      """Custom method for this class"""
      print("I'm a Pizza with Toppings!")

  # More complex class with multiple conditions
  class NonVegetarianPizza(PizzaWithToppings):
    """Pizza with meat or fish toppings"""
    equivalent_to = [
      # PizzaWithToppings AND (has MeatTopping OR has FishTopping)
      PizzaWithToppings & (
        onto.has_topping.some(onto.MeatTopping) | 
        onto.has_topping.some(onto.FishTopping)
      )
    ]
    
    def who(self):
      print("I'm a Non Vegetarian Pizza!")

In [8]:
# Create instances of new classes
fishPizza = NonVegetarianPizza("fishPizza", 
                               has_topping=[onto.FishTopping()])

pizzaHut = Restaurant("PizzaHut", servesPizza=[fishPizza])

# Verify Restaurant instances
print("Restaurant instances:", list(Restaurant.instances()))

# Access the pizzas served by PizzaHut
print("PizzaHut serves:", pizzaHut.servesPizza)

Restaurant instances: [pizza_onto.PizzaHut]
PizzaHut serves: [pizza_onto.fishPizza]


## Reasoning (Inference)

In [9]:
# Before reasoning: my_pizza is just classified as Pizza
print("Before reasoning:", my_pizza.__class__)

Before reasoning: pizza_onto.Pizza


In [10]:
# Run HermiT reasoner to infer new facts
# Reasoner analyzes ontology axioms and class definitions
# to derive new classification and relationships
# NOTE: Requires Java runtime (configure path on Windows if needed)
# owlready2.JAVA_EXE = "C:\\path\\to\\java.exe"
with onto:
  sync_reasoner()

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /Users/studio20/Downloads/.venv/lib/python3.14/site-packages/owlready2/hermit:/Users/studio20/Downloads/.venv/lib/python3.14/site-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////var/folders/xp/lbjy3zj90gq227t46d86bz4m0000gn/T/tmpb8q5qa06
* Owlready2 * HermiT took 1.0595240592956543 seconds
* Owlready * Reparenting pizza_onto.my_perfect_pizza: {pizza_onto.Pizza} => {pizza_onto.NonVegetarianPizza}
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)


In [11]:
# After reasoning: my_pizza is reclassified based on its properties
# Since it has toppings, reasoner infers it's a PizzawithToppings
# Since it has MeatTopping, reasoner infers it's NonVegetarianPizza
print("After reasoning:", my_pizza.__class__)

After reasoning: pizza_onto.NonVegetarianPizza


In [12]:
# Now we can call methods from the inferred class
my_pizza.who()

I'm a Non Vegetarian Pizza!


## Reasoning Inconsistent Ontology

* In case of inconsistent ontology, an OwlReadyInconsistentOntologyError is raised.


In [13]:
# Create an inconsistent class definition
# Inconsistency: Pizza cannot have Pizza as topping (circular)
with onto:
  class InconsistentPizza(PizzaWithToppings):
    """Invalid: Pizza with Pizza as topping"""
    equivalent_to = [
      PizzaWithToppings & (onto.has_topping.some(onto.Pizza))
    ]

* Inconcistent classes may occur without making the entire ontology inconsistent, as long as these classes have no individuals.
* Inconsistent classes are inferred as equivalent to Nothing (empty set).

In [14]:
# Create instance of inconsistent class
# This makes the ontology inconsistent
my_pizza2 = InconsistentPizza("Pizza2", has_topping = [onto.Pizza()])
print("Before reasoning:", my_pizza2.__class__)
print("Toppings:", my_pizza2.has_topping)

Before reasoning: pizza_onto.InconsistentPizza
Toppings: [pizza_onto.pizza1]


* Apply reasoner again and notice the produced error OwlReadyInconsistentOntologyError.

### Resolve Inconsistency

* Ontology editors like [Protégé](https://protege.stanford.edu/) can provide reasons for the inconsistency.
* Inconsistencies can also be found by searching for instances of inconsistent classes and removing them.

In [15]:
# Method 1: Check if class is equivalent to Nothing (empty set)
if Nothing in InconsistentPizza.equivalent_to:
  print("InconsistentPizza is inconsistent!")

In [16]:
# Method 2: List all inconsistent classes in the ontology
inc_classes = list(default_world.inconsistent_classes())
print("Inconsistent classes:", inc_classes)

Inconsistent classes: []


In [ ]:
# Remove instances of inconsistent classes
for inc_class in inc_classes:
  print(f"Removing instances of {inc_class}")

  for instance in onto.get_instances_of(inc_class):
    print(f"  Removing {instance}")
    
    destroy_entity(instance)  # Delete from ontology

* Apply reasoner again and notice the results.

## The Nothing Class

* This class exists in the ontology as concept only without any instances.
* To keep the ontology sound, this class must not be instantiated.

In [18]:
# Nothing represents the empty set in ontology
# It's a conceptual class with no instances
print("Nothing class type:", Nothing.__class__)

Nothing class type: <class 'owlready2.entity.ThingClass'>


In [19]:
# WARNING: Creating Nothing instances makes ontology inconsistent
# This should NOT be done in production code
with onto:
  nothing = Nothing("Empty")  # DO NOT DO THIS

In [20]:
# Verify (incorrectly) created instance
print("All individuals (including Nothing):", list(onto.individuals()))

All individuals (including Nothing): [pizza_onto.cheesetopping1, pizza_onto.meattopping1, pizza_onto.my_perfect_pizza, pizza_onto.fishtopping1, pizza_onto.fishPizza, pizza_onto.PizzaHut, pizza_onto.fishtopping2, pizza_onto.pizza1, pizza_onto.Pizza2, pizza_onto.Empty]


## Save the Ontology

In [21]:
# Save modified ontology to file in RDF/XML format
# Note: OWL/XML format not yet supported for writing
onto.save(file = "new_pizza_onto.owl", format = "rdfxml")
print("Ontology saved to new_pizza_onto.owl")

Ontology saved to new_pizza_onto.owl


In [23]:
# Exercise 1: Find all vegetarian pizzas
print("\n=== Exercise 1: Vegetarian Pizzas ===")
vegetarian_pizzas = onto.search(
  is_a=onto.Pizza,
  has_topping=onto.search(is_a=onto.VegetableTopping)
)
print("Vegetarian pizzas:", vegetarian_pizzas)

# Exercise 2: Create a custom pizza type
print("\n=== Exercise 2: Custom Pizza Type ===")
with onto:
  class SpicyPizza(PizzaWithToppings):
    """Pizza with spicy toppings"""
    equivalent_to = [
      PizzaWithToppings & onto.has_topping.some(onto.SpicyTopping)
    ]
        
  # If SpicyTopping doesn't exist, create it
  if not hasattr(onto, 'SpicyTopping'):
    class SpicyTopping(onto.Topping):
      pass

spicy = SpicyPizza("jalapeño_pizza", has_topping=[SpicyTopping()])
print("Created:", spicy)

# Exercise 3: Query restaurants and their pizzas
print("\n=== Exercise 3: Restaurant-Pizza Relationships ===")
for restaurant in Restaurant.instances():
  print(f"{restaurant.name} serves:")
  for pizza in restaurant.servesPizza:
    print(f"  - {pizza.name}")

# Exercise 4: Count pizzas by topping type
print("\n=== Exercise 4: Pizza Statistics ===")
topping_types = {}
for pizza in onto.Pizza.instances():
  for topping in pizza.has_topping:
    topping_class = topping.__class__.__name__
    topping_types[topping_class] = topping_types.get(topping_class, 0) + 1

for topping, count in topping_types.items():
  print(f"{topping}: {count} pizzas")


=== Exercise 1: Vegetarian Pizzas ===


AttributeError: 'NoneType' object has no attribute 'storid'